# 📄 Document Image Enhancement
## Text Enhancement through Deblurring and Denoising

This notebook provides a comprehensive pipeline for enhancing scanned document images.  
It covers classical image processing techniques as well as a deep learning approach using a **U-Net** architecture.

**Designed for:** Google Colab (GPU recommended)  
**Dataset storage:** Google Drive

---

### 📁 Dataset Setup Instructions

Before running this notebook, upload your dataset to Google Drive with the following structure:

```
MyDrive/
└── document_dataset/
    ├── degraded/    ← blurry/noisy input images (.jpg, .png, .tiff, .bmp)
    └── clean/       ← ground truth clean images (optional, for paired training)
```

If you only have unpaired images, place them in the `degraded/` folder and leave `clean/` empty or omit it.

---
## 1. Environment Setup & Google Drive Mount

In [ ]:
# Install any libraries not pre-installed on Google Colab
!pip install -q scikit-image opencv-python-headless Pillow

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive mounted successfully.')

In [ ]:
import os
import glob
import warnings
import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image
from scipy.signal import wiener

from skimage import img_as_float, img_as_ubyte
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim
from skimage.restoration import denoise_nl_means, estimate_sigma
from skimage.morphology import disk

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

warnings.filterwarnings('ignore')

# ── GPU detection ──────────────────────────────────────────────────────────────
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f'✅ GPU detected: {gpus[0].name}')
    # Allow GPU memory growth to avoid allocating all VRAM up front
    tf.config.experimental.set_memory_growth(gpus[0], True)
else:
    print('⚠️  No GPU detected – running on CPU. Training will be slow.')
    print('   Tip: In Colab, go to Runtime → Change runtime type → GPU')

print('\n📦 Library versions:')
print(f'   TensorFlow : {tf.__version__}')
print(f'   OpenCV     : {cv2.__version__}')
print(f'   NumPy      : {np.__version__}')

In [ ]:
# ── Path configuration ─────────────────────────────────────────────────────────
DRIVE_ROOT       = '/content/drive/MyDrive'
DATASET_DIR      = os.path.join(DRIVE_ROOT, 'document_dataset')
DEGRADED_DIR     = os.path.join(DATASET_DIR, 'degraded')
CLEAN_DIR        = os.path.join(DATASET_DIR, 'clean')
OUTPUT_DIR       = os.path.join(DRIVE_ROOT, 'document_enhancement_results')
MODEL_SAVE_PATH  = os.path.join(DRIVE_ROOT, 'unet_document_enhancer.keras')

# Create output directory if it does not exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Supported image formats
IMAGE_EXTENSIONS = ('*.jpg', '*.jpeg', '*.png', '*.tiff', '*.tif', '*.bmp')

print(f'Dataset (degraded) : {DEGRADED_DIR}')
print(f'Dataset (clean)    : {CLEAN_DIR}')
print(f'Results saved to   : {OUTPUT_DIR}')

---
## 2. Dataset Loading & Exploration

In [ ]:
def collect_image_paths(folder):
    """Return sorted list of image file paths found in *folder*."""
    paths = []
    for ext in IMAGE_EXTENSIONS:
        paths.extend(glob.glob(os.path.join(folder, ext)))
    return sorted(paths)


def load_image(path, as_gray=False):
    """Load an image from *path* using OpenCV; returns an RGB (or grayscale) NumPy array."""
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE if as_gray else cv2.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(f'Could not read image: {path}')
    if not as_gray:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)  # Convert BGR → RGB
    return img


# ── Collect paths ──────────────────────────────────────────────────────────────
degraded_paths = collect_image_paths(DEGRADED_DIR) if os.path.isdir(DEGRADED_DIR) else []
clean_paths    = collect_image_paths(CLEAN_DIR)    if os.path.isdir(CLEAN_DIR)    else []

paired_mode = len(clean_paths) > 0

print(f'Degraded images : {len(degraded_paths)}')
print(f'Clean images    : {len(clean_paths)}')
print(f'Mode            : {"paired" if paired_mode else "unpaired"}')

# ── Fallback: generate synthetic samples when no real images are available ─────
if len(degraded_paths) == 0:
    print('\n⚠️  No images found in the dataset folder.')
    print('   Generating synthetic document images for demonstration…')

    os.makedirs(DEGRADED_DIR, exist_ok=True)
    os.makedirs(CLEAN_DIR,    exist_ok=True)

    rng = np.random.default_rng(42)
    for i in range(6):
        # Simulate a white page with dark text-like horizontal stripes
        clean = np.full((256, 256, 3), 240, dtype=np.uint8)
        for y in range(30, 240, 20):
            thickness = rng.integers(2, 5)
            clean[y:y+thickness, 20:236] = rng.integers(0, 60, (thickness, 216, 3))

        # Add Gaussian noise to create a degraded version
        noise   = rng.normal(0, 20, clean.shape).astype(np.float32)
        blurred = cv2.GaussianBlur(clean, (5, 5), 1.5)
        degraded = np.clip(blurred.astype(np.float32) + noise, 0, 255).astype(np.uint8)

        cv2.imwrite(os.path.join(DEGRADED_DIR, f'sample_{i:02d}.png'),
                    cv2.cvtColor(degraded, cv2.COLOR_RGB2BGR))
        cv2.imwrite(os.path.join(CLEAN_DIR, f'sample_{i:02d}.png'),
                    cv2.cvtColor(clean,    cv2.COLOR_RGB2BGR))

    degraded_paths = collect_image_paths(DEGRADED_DIR)
    clean_paths    = collect_image_paths(CLEAN_DIR)
    paired_mode    = True
    print(f'✅ {len(degraded_paths)} synthetic image pairs created.')

In [ ]:
def print_image_info(path):
    """Print metadata for the image at *path*."""
    img = load_image(path)
    h, w = img.shape[:2]
    c    = img.shape[2] if img.ndim == 3 else 1
    print(f'  {os.path.basename(path):30s}  size={w}x{h}  channels={c}  dtype={img.dtype}')


print('── Degraded image properties ──')
for p in degraded_paths[:5]:
    print_image_info(p)

if paired_mode:
    print('\n── Clean image properties ──')
    for p in clean_paths[:5]:
        print_image_info(p)

In [ ]:
def show_image_grid(paths, title, max_images=6, cmap=None):
    """Display up to *max_images* images in a grid."""
    n   = min(len(paths), max_images)
    fig, axes = plt.subplots(1, n, figsize=(3 * n, 3))
    if n == 1:
        axes = [axes]
    fig.suptitle(title, fontsize=14, fontweight='bold')
    for ax, p in zip(axes, paths[:n]):
        img = load_image(p)
        ax.imshow(img, cmap=cmap)
        ax.set_title(os.path.basename(p), fontsize=7)
        ax.axis('off')
    plt.tight_layout()
    plt.show()


show_image_grid(degraded_paths, 'Sample Degraded Images')
if paired_mode:
    show_image_grid(clean_paths, 'Sample Clean (Ground-Truth) Images')

---
## 3. Image Degradation Simulation

These functions can be used to artificially degrade clean images, thereby generating paired training data when only clean originals are available.

In [ ]:
rng = np.random.default_rng(0)


def add_gaussian_noise(image, sigma=25):
    """Add zero-mean Gaussian noise with standard deviation *sigma*."""
    noise   = rng.normal(0, sigma, image.shape).astype(np.float32)
    noisy   = np.clip(image.astype(np.float32) + noise, 0, 255)
    return noisy.astype(np.uint8)


def add_salt_pepper_noise(image, density=0.05):
    """Add salt-and-pepper noise with total *density* (fraction of pixels affected)."""
    out  = image.copy()
    mask = rng.random(image.shape[:2])
    # Salt (white pixels)
    out[mask < density / 2]                   = 255
    # Pepper (black pixels)
    out[(mask >= density / 2) & (mask < density)] = 0
    return out


def apply_gaussian_blur(image, kernel_size=5, sigma=2.0):
    """Apply Gaussian blur with given *kernel_size* (must be odd) and *sigma*."""
    k = kernel_size if kernel_size % 2 == 1 else kernel_size + 1
    return cv2.GaussianBlur(image, (k, k), sigma)


def apply_motion_blur(image, kernel_size=15, angle=0):
    """Simulate motion blur along *angle* degrees."""
    kernel = np.zeros((kernel_size, kernel_size), dtype=np.float32)
    kernel[kernel_size // 2, :] = 1.0 / kernel_size

    # Rotate kernel to the desired angle
    M      = cv2.getRotationMatrix2D((kernel_size // 2, kernel_size // 2), angle, 1)
    kernel = cv2.warpAffine(kernel, M, (kernel_size, kernel_size))
    kernel /= kernel.sum() + 1e-8  # Re-normalise after rotation

    return cv2.filter2D(image, -1, kernel)


def apply_combined_degradation(image, blur_k=5, blur_sigma=1.5, noise_sigma=20, sp_density=0.02):
    """Apply Gaussian blur + Gaussian noise + salt-and-pepper noise in sequence."""
    img = apply_gaussian_blur(image, blur_k, blur_sigma)
    img = add_gaussian_noise(img, noise_sigma)
    img = add_salt_pepper_noise(img, sp_density)
    return img


print('✅ Degradation functions defined.')

In [ ]:
# Visualise original vs. each degradation type
sample_path = degraded_paths[0] if not paired_mode else clean_paths[0]
original    = load_image(sample_path)

degradations = {
    'Original'             : original,
    'Gaussian Noise σ=25'  : add_gaussian_noise(original, sigma=25),
    'Salt & Pepper 5%'     : add_salt_pepper_noise(original, density=0.05),
    'Gaussian Blur k=9'    : apply_gaussian_blur(original, kernel_size=9, sigma=3),
    'Motion Blur 15px 45°' : apply_motion_blur(original, kernel_size=15, angle=45),
    'Combined'             : apply_combined_degradation(original),
}

fig, axes = plt.subplots(1, len(degradations), figsize=(18, 3))
fig.suptitle('Degradation Simulation', fontsize=14, fontweight='bold')
for ax, (title, img) in zip(axes, degradations.items()):
    ax.imshow(img)
    ax.set_title(title, fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()

---
## 4. Classical / Traditional Enhancement Methods

### 4a. Denoising

In [ ]:
def denoise_gaussian(image, kernel_size=5):
    """Gaussian low-pass filter for noise suppression."""
    k = kernel_size if kernel_size % 2 == 1 else kernel_size + 1
    return cv2.GaussianBlur(image, (k, k), 0)


def denoise_median(image, kernel_size=3):
    """Median filter – effective against salt-and-pepper noise."""
    k = kernel_size if kernel_size % 2 == 1 else kernel_size + 1
    return cv2.medianBlur(image, k)


def denoise_bilateral(image, d=9, sigma_color=75, sigma_space=75):
    """Bilateral filter – denoises while preserving edges."""
    return cv2.bilateralFilter(image, d, sigma_color, sigma_space)


def denoise_nlm(image, h=10, fast=True):
    """Non-local means denoising (OpenCV fastNlMeansDenoising*)."""
    if image.ndim == 2:
        return cv2.fastNlMeansDenoising(image, None, h, 7, 21)
    return cv2.fastNlMeansDenoisingColored(image, None, h, h, 7, 21)


def denoise_morphological(image, operation='opening', radius=1):
    """Remove noise using morphological opening or closing.

    - Opening (erosion → dilation): removes small bright specks (salt noise).
    - Closing (dilation → erosion): fills small dark holes (pepper noise).
    """
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE,
                                       (2 * radius + 1, 2 * radius + 1))
    op     = cv2.MORPH_OPEN if operation == 'opening' else cv2.MORPH_CLOSE
    return cv2.morphologyEx(image, op, kernel)


# ── Visualise all denoising methods ───────────────────────────────────────────
noisy    = add_gaussian_noise(original, sigma=25)
sp_noisy = add_salt_pepper_noise(original, density=0.05)

denoised_results = {
    'Noisy Input'       : noisy,
    'Gaussian Filter'   : denoise_gaussian(noisy),
    'Median Filter'     : denoise_median(sp_noisy),
    'Bilateral Filter'  : denoise_bilateral(noisy),
    'Non-local Means'   : denoise_nlm(noisy),
    'Morph. Opening'    : denoise_morphological(sp_noisy, 'opening'),
    'Morph. Closing'    : denoise_morphological(sp_noisy, 'closing'),
}

fig, axes = plt.subplots(1, len(denoised_results), figsize=(21, 3))
fig.suptitle('Denoising Methods Comparison', fontsize=14, fontweight='bold')
for ax, (title, img) in zip(axes, denoised_results.items()):
    ax.imshow(img)
    ax.set_title(title, fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()

### 4b. Deblurring

In [ ]:
def sharpen_unsharp_mask(image, sigma=1.0, amount=1.5, threshold=0):
    """Unsharp masking: enhanced = original + amount * (original - blurred)."""
    blurred = cv2.GaussianBlur(image, (0, 0), sigma)
    sharp   = cv2.addWeighted(image, 1 + amount, blurred, -amount, 0)
    if threshold > 0:
        diff = np.abs(image.astype(np.int16) - blurred.astype(np.int16))
        mask = diff < threshold
        sharp[mask] = image[mask]
    return sharp


def sharpen_wiener(image, noise_var=0.01):
    """Wiener filter (channel-wise) via scipy.signal.wiener."""
    img_f = img_as_float(image)
    if img_f.ndim == 2:
        filtered = wiener(img_f, noise=noise_var)
    else:
        filtered = np.stack(
            [wiener(img_f[:, :, c], noise=noise_var) for c in range(img_f.shape[2])],
            axis=2
        )
    return img_as_ubyte(np.clip(filtered, 0, 1))


def sharpen_laplacian(image, alpha=0.7):
    """Laplacian sharpening: enhanced = original - alpha * laplacian(original)."""
    if image.ndim == 3:
        gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    else:
        gray = image
    lap  = cv2.Laplacian(gray, cv2.CV_64F)
    if image.ndim == 3:
        lap = np.stack([lap] * 3, axis=2)
    sharp = np.clip(image.astype(np.float64) - alpha * lap, 0, 255)
    return sharp.astype(np.uint8)


# ── Visualise deblurring methods ───────────────────────────────────────────────
blurry = apply_gaussian_blur(original, kernel_size=9, sigma=3)

deblur_results = {
    'Blurry Input'   : blurry,
    'Unsharp Mask'   : sharpen_unsharp_mask(blurry),
    'Wiener Filter'  : sharpen_wiener(blurry),
    'Laplacian Sharp': sharpen_laplacian(blurry),
}

fig, axes = plt.subplots(1, len(deblur_results), figsize=(12, 3))
fig.suptitle('Deblurring Methods Comparison', fontsize=14, fontweight='bold')
for ax, (title, img) in zip(axes, deblur_results.items()):
    ax.imshow(img)
    ax.set_title(title, fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

### 4c. Text-Specific Enhancement

In [ ]:
def to_gray(image):
    """Convert RGB image to grayscale if needed."""
    return cv2.cvtColor(image, cv2.COLOR_RGB2GRAY) if image.ndim == 3 else image


def adaptive_threshold_gaussian(image, block_size=11, C=2):
    """Gaussian adaptive thresholding – handles uneven illumination."""
    gray = to_gray(image)
    b    = block_size if block_size % 2 == 1 else block_size + 1
    return cv2.adaptiveThreshold(gray, 255,
                                 cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                 cv2.THRESH_BINARY, b, C)


def adaptive_threshold_mean(image, block_size=11, C=2):
    """Mean adaptive thresholding."""
    gray = to_gray(image)
    b    = block_size if block_size % 2 == 1 else block_size + 1
    return cv2.adaptiveThreshold(gray, 255,
                                 cv2.ADAPTIVE_THRESH_MEAN_C,
                                 cv2.THRESH_BINARY, b, C)


def otsu_binarize(image):
    """Otsu's global binarization – optimal threshold found automatically."""
    gray = to_gray(image)
    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return binary


def apply_clahe(image, clip_limit=2.0, tile_grid_size=(8, 8)):
    """CLAHE (Contrast Limited Adaptive Histogram Equalization) on the L channel."""
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
    if image.ndim == 3:
        lab         = cv2.cvtColor(image, cv2.COLOR_RGB2LAB)
        lab[:, :, 0] = clahe.apply(lab[:, :, 0])
        return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)
    return clahe.apply(image)


def contrast_stretch(image, p_low=2, p_high=98):
    """Stretch contrast by clipping to [p_low, p_high] percentile values."""
    lo = np.percentile(image, p_low)
    hi = np.percentile(image, p_high)
    stretched = np.clip((image.astype(np.float32) - lo) / (hi - lo + 1e-8) * 255, 0, 255)
    return stretched.astype(np.uint8)


# ── Visualise text-specific methods ───────────────────────────────────────────
sample_degraded = load_image(degraded_paths[0])

text_results = {
    'Input'                : sample_degraded,
    'Adaptive Thresh (G)'  : adaptive_threshold_gaussian(sample_degraded),
    'Adaptive Thresh (M)'  : adaptive_threshold_mean(sample_degraded),
    "Otsu's Binarize"      : otsu_binarize(sample_degraded),
    'CLAHE'                : apply_clahe(sample_degraded),
    'Contrast Stretch'     : contrast_stretch(sample_degraded),
}

fig, axes = plt.subplots(1, len(text_results), figsize=(18, 3))
fig.suptitle('Text-Specific Enhancement Methods', fontsize=14, fontweight='bold')
for ax, (title, img) in zip(axes, text_results.items()):
    # Grayscale images need cmap='gray'
    cmap = 'gray' if img.ndim == 2 else None
    ax.imshow(img, cmap=cmap)
    ax.set_title(title, fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()

---
## 5. Deep Learning Model — U-Net for Document Enhancement

The **U-Net** architecture is an encoder-decoder network with skip connections.  
It was originally designed for biomedical image segmentation but excels at any pixel-to-pixel mapping task, including document image restoration.

```
Input Image
    │
    ├─ Encoder (contracting path) ─ captures context via downsampling
    │       enc1 → enc2 → enc3 → enc4 → bottleneck
    │
    └─ Decoder (expansive path)   ─ enables precise localisation via upsampling
            dec4 → dec3 → dec2 → dec1 → Output Image
              ↑      ↑      ↑      ↑
           skip connections from encoder
```

### 5a. Model Architecture

In [ ]:
def conv_block(x, filters, kernel_size=3, activation='relu', batch_norm=True):
    """Two consecutive Conv2D + optional BatchNorm + Activation layers."""
    x = layers.Conv2D(filters, kernel_size, padding='same', use_bias=not batch_norm)(x)
    if batch_norm:
        x = layers.BatchNormalization()(x)
    x = layers.Activation(activation)(x)

    x = layers.Conv2D(filters, kernel_size, padding='same', use_bias=not batch_norm)(x)
    if batch_norm:
        x = layers.BatchNormalization()(x)
    x = layers.Activation(activation)(x)
    return x


def build_unet(input_shape=(256, 256, 3), base_filters=32):
    """Build and return a U-Net model for image-to-image mapping.

    Args:
        input_shape   : (H, W, C) tuple for the input image.
        base_filters  : Number of filters in the first encoder block;
                        doubles with each downsampling level.

    Returns:
        A compiled Keras Model.
    """
    inputs = layers.Input(shape=input_shape, name='degraded_input')

    # ── Encoder ───────────────────────────────────────────────────────────────
    e1 = conv_block(inputs,      base_filters)        # 256×256×32
    p1 = layers.MaxPooling2D()(e1)                    # 128×128×32

    e2 = conv_block(p1,      base_filters * 2)        # 128×128×64
    p2 = layers.MaxPooling2D()(e2)                    # 64×64×64

    e3 = conv_block(p2,      base_filters * 4)        # 64×64×128
    p3 = layers.MaxPooling2D()(e3)                    # 32×32×128

    e4 = conv_block(p3,      base_filters * 8)        # 32×32×256
    p4 = layers.MaxPooling2D()(e4)                    # 16×16×256

    # ── Bottleneck ────────────────────────────────────────────────────────────
    b  = conv_block(p4,      base_filters * 16)       # 16×16×512
    b  = layers.Dropout(0.3)(b)

    # ── Decoder ───────────────────────────────────────────────────────────────
    u4 = layers.Conv2DTranspose(base_filters * 8,  2, strides=2, padding='same')(b)
    u4 = layers.Concatenate()([u4, e4])               # Skip connection
    d4 = conv_block(u4, base_filters * 8)

    u3 = layers.Conv2DTranspose(base_filters * 4,  2, strides=2, padding='same')(d4)
    u3 = layers.Concatenate()([u3, e3])
    d3 = conv_block(u3, base_filters * 4)

    u2 = layers.Conv2DTranspose(base_filters * 2,  2, strides=2, padding='same')(d3)
    u2 = layers.Concatenate()([u2, e2])
    d2 = conv_block(u2, base_filters * 2)

    u1 = layers.Conv2DTranspose(base_filters,      2, strides=2, padding='same')(d2)
    u1 = layers.Concatenate()([u1, e1])
    d1 = conv_block(u1, base_filters)

    # ── Output (sigmoid → pixel values in [0, 1]) ─────────────────────────────
    outputs = layers.Conv2D(input_shape[2], 1, activation='sigmoid', name='enhanced_output')(d1)

    model = Model(inputs, outputs, name='UNet_DocumentEnhancer')
    return model


# ── Instantiate and summarise the model ───────────────────────────────────────
INPUT_SHAPE  = (256, 256, 3)
unet         = build_unet(input_shape=INPUT_SHAPE, base_filters=32)
unet.summary()

### 5b. Loss Function — MSE + SSIM

In [ ]:
def ssim_loss(y_true, y_pred):
    """1 - mean SSIM across a batch of images (maximising SSIM = minimising this)."""
    return 1.0 - tf.reduce_mean(
        tf.image.ssim(y_true, y_pred, max_val=1.0)
    )


def combined_loss(y_true, y_pred, alpha=0.84):
    """Weighted combination: alpha * SSIM_loss + (1 - alpha) * MSE.

    alpha=0.84 follows the recommendation in SSIM-based perceptual loss literature.
    """
    mse  = tf.reduce_mean(tf.square(y_true - y_pred))
    return alpha * ssim_loss(y_true, y_pred) + (1.0 - alpha) * mse


# ── Compile the model ─────────────────────────────────────────────────────────
unet.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss=combined_loss,
    metrics=['mae']
)
print('✅ Model compiled with combined MSE + SSIM loss.')

### 5c. Data Generator

In [ ]:
class DocumentDataGenerator(keras.utils.Sequence):
    """Keras-compatible data generator that loads image pairs from Google Drive.

    Each batch yields:
        X : degraded images  – shape (batch, H, W, C)  float32 in [0, 1]
        y : clean images     – shape (batch, H, W, C)  float32 in [0, 1]
    """

    def __init__(self, degraded_paths, clean_paths, image_size=(256, 256),
                 batch_size=8, augment=False, shuffle=True):
        self.degraded_paths = degraded_paths
        self.clean_paths    = clean_paths
        self.image_size     = image_size
        self.batch_size     = batch_size
        self.augment        = augment
        self.shuffle        = shuffle
        self._rng           = np.random.default_rng()
        self.indices        = np.arange(len(degraded_paths))
        if self.shuffle:
            self._rng.shuffle(self.indices)

    def __len__(self):
        return max(1, -(-len(self.degraded_paths) // self.batch_size))

    def __getitem__(self, idx):
        batch_idx = self.indices[idx * self.batch_size:(idx + 1) * self.batch_size]
        X, y = [], []
        for i in batch_idx:
            deg  = self._load(self.degraded_paths[i])
            cln  = self._load(self.clean_paths[i])
            if self.augment:
                deg, cln = self._augment(deg, cln)
            X.append(deg)
            y.append(cln)
        return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

    def on_epoch_end(self):
        if self.shuffle:
            self._rng.shuffle(self.indices)

    def _load(self, path):
        """Load, resize, and normalise an image to [0, 1]."""
        img = cv2.imread(path, cv2.IMREAD_COLOR)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, self.image_size, interpolation=cv2.INTER_AREA)
        return img.astype(np.float32) / 255.0

    def _augment(self, deg, cln):
        """Apply identical random flips to both images."""
        if self._rng.random() > 0.5:          # Horizontal flip
            deg, cln = deg[:, ::-1], cln[:, ::-1]
        if self._rng.random() > 0.5:          # Vertical flip
            deg, cln = deg[::-1], cln[::-1]
        return deg, cln


print('✅ DocumentDataGenerator defined.')

### 5d. Training

In [ ]:
# ── Split dataset into train / validation ─────────────────────────────────────
# Requires a paired dataset; falls back gracefully when clean_paths is empty.

if paired_mode and len(degraded_paths) >= 2:
    # Ensure lists have matching lengths
    n       = min(len(degraded_paths), len(clean_paths))
    d_paths = degraded_paths[:n]
    c_paths = clean_paths[:n]

    split        = max(1, int(0.8 * n))
    train_d, val_d = d_paths[:split], d_paths[split:]
    train_c, val_c = c_paths[:split], c_paths[split:]

    BATCH_SIZE = 4
    train_gen  = DocumentDataGenerator(train_d, train_c, batch_size=BATCH_SIZE, augment=True)
    val_gen    = DocumentDataGenerator(val_d,   val_c,   batch_size=BATCH_SIZE, augment=False)

    print(f'Training samples   : {len(train_d)}')
    print(f'Validation samples : {len(val_d)}')
    print(f'Batch size         : {BATCH_SIZE}')
    print(f'Train steps/epoch  : {len(train_gen)}')

    # ── Callbacks ─────────────────────────────────────────────────────────────
    callbacks = [
        ModelCheckpoint(
            filepath=MODEL_SAVE_PATH,
            monitor='val_loss',
            save_best_only=True,
            verbose=1
        ),
        EarlyStopping(
            monitor='val_loss',
            patience=10,
            restore_best_weights=True,
            verbose=1
        ),
        ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=5,
            min_lr=1e-6,
            verbose=1
        ),
    ]

    EPOCHS = 50  # Increase for a real dataset
    print(f'\n🚀 Starting training for up to {EPOCHS} epochs…')

    history = unet.fit(
        train_gen,
        validation_data=val_gen,
        epochs=EPOCHS,
        callbacks=callbacks,
        verbose=1
    )

    print(f'\n✅ Training complete. Best model saved to: {MODEL_SAVE_PATH}')

else:
    print('⚠️  Skipping training: need at least 2 paired image samples.')
    print('   Upload more images to DEGRADED_DIR and CLEAN_DIR and re-run.')
    history = None

In [ ]:
# ── Plot training history ─────────────────────────────────────────────────────
if history is not None:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle('Training History', fontsize=14, fontweight='bold')

    axes[0].plot(history.history['loss'],     label='Train Loss')
    axes[0].plot(history.history['val_loss'], label='Val Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Loss (MSE + SSIM)')
    axes[0].legend()
    axes[0].grid(True)

    axes[1].plot(history.history['mae'],     label='Train MAE')
    axes[1].plot(history.history['val_mae'], label='Val MAE')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('MAE')
    axes[1].set_title('Mean Absolute Error')
    axes[1].legend()
    axes[1].grid(True)

    plt.tight_layout()
    plt.show()
else:
    print('No training history available – skipping plot.')

### 5e. Load Saved Model & Run Inference

In [ ]:
def preprocess_for_unet(image, target_size=(256, 256)):
    """Resize and normalise a uint8 RGB image for U-Net inference."""
    resized = cv2.resize(image, target_size, interpolation=cv2.INTER_AREA)
    return resized.astype(np.float32) / 255.0


def unet_enhance(model, image):
    """Run U-Net inference on *image* (uint8 RGB).

    Returns the enhanced image as uint8 RGB at the original spatial resolution.
    """
    h, w = image.shape[:2]
    inp  = preprocess_for_unet(image)[np.newaxis]   # (1, 256, 256, 3)
    pred = model.predict(inp, verbose=0)[0]          # (256, 256, 3)
    out  = np.clip(pred * 255, 0, 255).astype(np.uint8)
    # Restore original resolution
    return cv2.resize(out, (w, h), interpolation=cv2.INTER_LINEAR)


# Load the best saved model (if it exists)
if os.path.exists(MODEL_SAVE_PATH):
    print(f'Loading model from {MODEL_SAVE_PATH}…')
    unet = keras.models.load_model(
        MODEL_SAVE_PATH,
        custom_objects={'combined_loss': combined_loss, 'ssim_loss': ssim_loss}
    )
    print('✅ Model loaded.')
else:
    print(f'⚠️  No saved model found at {MODEL_SAVE_PATH}.')
    print('   Using the current (potentially untrained) model weights for demonstration.')

# ── Run inference on the first test image ─────────────────────────────────────
test_img      = load_image(degraded_paths[0])
unet_enhanced = unet_enhance(unet, test_img)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.imshow(test_img);      ax1.set_title('Degraded Input');  ax1.axis('off')
ax2.imshow(unet_enhanced); ax2.set_title('U-Net Enhanced'); ax2.axis('off')
plt.suptitle('U-Net Inference Result', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 6. Evaluation Metrics

We compute **PSNR** (Peak Signal-to-Noise Ratio) and **SSIM** (Structural Similarity Index) to quantify the quality of each enhancement method relative to the clean reference.

In [ ]:
def compute_psnr(reference, enhanced):
    """Compute PSNR in dB between *reference* and *enhanced* (uint8 arrays)."""
    ref = reference.astype(np.float64) / 255.0
    enh = enhanced.astype(np.float64) / 255.0
    # Handle possible channel mismatch (e.g. grayscale output vs colour reference)
    if ref.ndim != enh.ndim:
        if ref.ndim == 3 and enh.ndim == 2:
            ref = cv2.cvtColor((ref * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY).astype(np.float64) / 255.0
        elif ref.ndim == 2 and enh.ndim == 3:
            enh = cv2.cvtColor((enh * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY).astype(np.float64) / 255.0
    return psnr(ref, enh, data_range=1.0)


def compute_ssim(reference, enhanced):
    """Compute mean SSIM between *reference* and *enhanced* (uint8 arrays)."""
    ref = reference.astype(np.float64) / 255.0
    enh = enhanced.astype(np.float64) / 255.0
    if ref.ndim != enh.ndim:
        if ref.ndim == 3 and enh.ndim == 2:
            ref = cv2.cvtColor((ref * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY).astype(np.float64) / 255.0
        elif ref.ndim == 2 and enh.ndim == 3:
            enh = cv2.cvtColor((enh * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY).astype(np.float64) / 255.0
    channel_axis = 2 if ref.ndim == 3 else None
    return ssim(ref, enh, data_range=1.0, channel_axis=channel_axis)


print('✅ Evaluation metric functions defined.')

In [ ]:
# ── Prepare test image and reference ──────────────────────────────────────────
if paired_mode:
    reference_img = load_image(clean_paths[0])
    test_img      = load_image(degraded_paths[0])
else:
    # Without a clean reference, measure degradation effect only
    test_img      = load_image(degraded_paths[0])
    reference_img = test_img.copy()   # trivial reference
    print('⚠️  No clean reference – PSNR/SSIM measured against the input image itself.')

# Align sizes
h, w = reference_img.shape[:2]
test_resized = cv2.resize(test_img, (w, h))

# ── Apply all methods ─────────────────────────────────────────────────────────
methods = {
    'Noisy Input (baseline)' : test_resized,
    'Gaussian Filter'        : denoise_gaussian(test_resized),
    'Median Filter'          : denoise_median(test_resized),
    'Bilateral Filter'       : denoise_bilateral(test_resized),
    'Non-local Means'        : denoise_nlm(test_resized),
    'Unsharp Mask'           : sharpen_unsharp_mask(test_resized),
    'Wiener Filter'          : sharpen_wiener(test_resized),
    'Laplacian Sharpening'   : sharpen_laplacian(test_resized),
    'CLAHE'                  : apply_clahe(test_resized),
    'Contrast Stretch'       : contrast_stretch(test_resized),
    'U-Net'                  : cv2.resize(unet_enhanced, (w, h)),
}

# ── Compute metrics ───────────────────────────────────────────────────────────
print(f'{"Method":<30}  {"PSNR (dB)":>10}  {"SSIM":>8}')
print('─' * 55)
metric_table = {}
for name, img in methods.items():
    p = compute_psnr(reference_img, img)
    s = compute_ssim(reference_img, img)
    metric_table[name] = (p, s)
    print(f'{name:<30}  {p:>10.2f}  {s:>8.4f}')

In [ ]:
# ── Bar chart comparison ───────────────────────────────────────────────────────
names  = list(metric_table.keys())
psnrs  = [v[0] for v in metric_table.values()]
ssims  = [v[1] for v in metric_table.values()]

x     = np.arange(len(names))
width = 0.4

fig, ax = plt.subplots(figsize=(14, 5))
bars1 = ax.bar(x - width / 2, psnrs, width, label='PSNR (dB)', color='steelblue')
ax.set_ylabel('PSNR (dB)')
ax.set_title('Method Comparison — PSNR & SSIM', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=35, ha='right', fontsize=8)
ax.legend(loc='upper left')
ax.grid(axis='y', alpha=0.4)

ax2   = ax.twinx()
bars2 = ax2.bar(x + width / 2, ssims, width, label='SSIM', color='coral', alpha=0.8)
ax2.set_ylabel('SSIM')
ax2.set_ylim(0, 1.2)
ax2.legend(loc='upper right')

plt.tight_layout()
plt.show()

---
## 7. Visualisation & Results

In [ ]:
def show_comparison(original, enhanced, title_orig='Original', title_enh='Enhanced', figsize=(10, 4)):
    """Side-by-side comparison of two images."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=figsize)
    ax1.imshow(original, cmap='gray' if original.ndim == 2 else None)
    ax1.set_title(title_orig, fontsize=11)
    ax1.axis('off')
    ax2.imshow(enhanced, cmap='gray' if enhanced.ndim == 2 else None)
    ax2.set_title(title_enh, fontsize=11)
    ax2.axis('off')
    plt.tight_layout()
    plt.show()


def show_zoomed_crop(image, region, title='Zoomed', figsize=(4, 4)):
    """Display a zoomed-in crop defined by *region* = (y1, y2, x1, x2)."""
    y1, y2, x1, x2 = region
    crop = image[y1:y2, x1:x2]
    plt.figure(figsize=figsize)
    plt.imshow(crop, cmap='gray' if crop.ndim == 2 else None)
    plt.title(title, fontsize=11)
    plt.axis('off')
    plt.show()


# ── Degraded → Best classical method → U-Net ──────────────────────────────────
best_classical_name = max(
    {k: v for k, v in metric_table.items() if k != 'U-Net' and k != 'Noisy Input (baseline)'},
    key=lambda k: metric_table[k][1]  # highest SSIM
)
best_classical_img  = methods[best_classical_name]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (title, img) in zip(axes, [
    ('Degraded Input',       test_resized),
    (f'Best Classical\n({best_classical_name})', best_classical_img),
    ('U-Net Enhanced',       methods['U-Net']),
]):
    ax.imshow(img, cmap='gray' if img.ndim == 2 else None)
    ax.set_title(title, fontsize=10)
    ax.axis('off')
plt.suptitle('Enhancement Pipeline Comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Full comparison grid across all methods ────────────────────────────────────
n_methods = len(methods)
n_cols    = 4
n_rows    = -(-n_methods // n_cols)  # Ceiling division

fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 4 * n_rows))
axes_flat = axes.flatten() if n_rows > 1 else axes
fig.suptitle('All Enhancement Methods — Comparison Grid', fontsize=14, fontweight='bold')

for ax, (name, img) in zip(axes_flat, methods.items()):
    p, s = metric_table[name]
    ax.imshow(img, cmap='gray' if img.ndim == 2 else None)
    ax.set_title(f'{name}\nPSNR={p:.1f}dB  SSIM={s:.3f}', fontsize=7)
    ax.axis('off')

# Hide any unused axes
for ax in axes_flat[n_methods:]:
    ax.set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# ── Zoomed-in crop to highlight text detail ────────────────────────────────────
h_ref, w_ref = test_resized.shape[:2]
cy, cx       = h_ref // 2, w_ref // 2
crop_size    = min(64, h_ref // 3, w_ref // 3)
region       = (cy - crop_size, cy + crop_size, cx - crop_size, cx + crop_size)

crop_pairs = [
    ('Degraded', test_resized),
    ('Bilateral Filter', methods['Bilateral Filter']),
    ('Non-local Means',  methods['Non-local Means']),
    ('Unsharp Mask',     methods['Unsharp Mask']),
    ('U-Net',            methods['U-Net']),
]
if paired_mode:
    crop_pairs.insert(0, ('Clean Reference', reference_img))

fig, axes = plt.subplots(1, len(crop_pairs), figsize=(3 * len(crop_pairs), 3))
fig.suptitle('Zoomed-In Text Crop Comparison', fontsize=13, fontweight='bold')
for ax, (title, img) in zip(axes, crop_pairs):
    y1, y2, x1, x2 = region
    crop = cv2.resize(img[y1:y2, x1:x2], (128, 128), interpolation=cv2.INTER_NEAREST)
    ax.imshow(crop, cmap='gray' if crop.ndim == 2 else None)
    ax.set_title(title, fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# ── Save enhanced results to Google Drive ─────────────────────────────────────
print('Saving enhanced images to Google Drive…')

for name, img in methods.items():
    safe_name = name.replace(' ', '_').replace('/', '_').replace('(', '').replace(')', '')
    save_path = os.path.join(OUTPUT_DIR, f'enhanced_{safe_name}.png')
    # Convert grayscale back to BGR for imwrite if necessary
    if img.ndim == 2:
        cv2.imwrite(save_path, img)
    else:
        cv2.imwrite(save_path, cv2.cvtColor(img, cv2.COLOR_RGB2BGR))
    print(f'  Saved: {save_path}')

print(f'\n✅ All results saved to {OUTPUT_DIR}')

---
## 8. Complete Pipeline Function

A single `enhance_document()` function that wraps the entire pipeline end-to-end.

In [ ]:
def enhance_document(
    image_path,
    method='bilateral',
    output_path=None,
    unet_model=None,
    show=True,
):
    """Enhance a document image using the selected method and (optionally) save the result.

    Args:
        image_path  : Path to the input (degraded) image.
        method      : Enhancement algorithm to apply. One of:
                        'gaussian'   – Gaussian denoising
                        'median'     – Median denoising
                        'bilateral'  – Bilateral denoising (default)
                        'nlm'        – Non-local means denoising
                        'unsharp'    – Unsharp masking (deblurring)
                        'wiener'     – Wiener filter (deblurring)
                        'laplacian'  – Laplacian sharpening
                        'clahe'      – CLAHE contrast enhancement
                        'otsu'       – Otsu's binarization
                        'adaptive_g' – Adaptive Gaussian thresholding
                        'stretch'    – Contrast stretching
                        'combined'   – Bilateral + Unsharp Mask + CLAHE
                        'unet'       – U-Net deep learning model
        output_path : If provided, the enhanced image is saved here.
        unet_model  : Keras model instance (required when method='unet').
        show        : Whether to display input and output side-by-side.

    Returns:
        enhanced (np.ndarray): The enhanced image as a uint8 NumPy array.
    """
    # ── Load ───────────────────────────────────────────────────────────────────
    if not os.path.isfile(image_path):
        raise FileNotFoundError(f'Input image not found: {image_path}')

    img = load_image(image_path)
    print(f'[enhance_document] Input  : {image_path}  ({img.shape[1]}×{img.shape[0]})')

    # ── Enhance ────────────────────────────────────────────────────────────────
    method = method.lower().strip()

    if method == 'gaussian':
        enhanced = denoise_gaussian(img)
    elif method == 'median':
        enhanced = denoise_median(img)
    elif method == 'bilateral':
        enhanced = denoise_bilateral(img)
    elif method == 'nlm':
        enhanced = denoise_nlm(img)
    elif method == 'unsharp':
        enhanced = sharpen_unsharp_mask(img)
    elif method == 'wiener':
        enhanced = sharpen_wiener(img)
    elif method == 'laplacian':
        enhanced = sharpen_laplacian(img)
    elif method == 'clahe':
        enhanced = apply_clahe(img)
    elif method == 'otsu':
        enhanced = otsu_binarize(img)
    elif method == 'adaptive_g':
        enhanced = adaptive_threshold_gaussian(img)
    elif method == 'stretch':
        enhanced = contrast_stretch(img)
    elif method == 'combined':
        # Multi-step pipeline: denoise → deblur → enhance contrast
        step1    = denoise_bilateral(img)
        step2    = sharpen_unsharp_mask(step1, sigma=1.0, amount=1.0)
        enhanced = apply_clahe(step2)
    elif method == 'unet':
        if unet_model is None:
            raise ValueError('unet_model must be provided when method="unet".')
        enhanced = unet_enhance(unet_model, img)
    else:
        raise ValueError(
            f'Unknown method: "{method}". Choose from: gaussian, median, bilateral, nlm, '
            f'unsharp, wiener, laplacian, clahe, otsu, adaptive_g, stretch, combined, unet.'
        )

    print(f'[enhance_document] Method : {method}')

    # ── Save ───────────────────────────────────────────────────────────────────
    if output_path is not None:
        os.makedirs(os.path.dirname(output_path) or '.', exist_ok=True)
        save_img = enhanced if enhanced.ndim == 2 else cv2.cvtColor(enhanced, cv2.COLOR_RGB2BGR)
        cv2.imwrite(output_path, save_img)
        print(f'[enhance_document] Saved  : {output_path}')

    # ── Display ────────────────────────────────────────────────────────────────
    if show:
        show_comparison(img, enhanced,
                        title_orig='Input (Degraded)',
                        title_enh=f'Enhanced ({method})')

    return enhanced


print('✅ enhance_document() pipeline function defined.')

In [ ]:
# ── Demo: run the pipeline on the first degraded image with each method ────────
demo_methods = ['bilateral', 'nlm', 'unsharp', 'clahe', 'combined']

for m in demo_methods:
    out_path = os.path.join(OUTPUT_DIR, f'pipeline_{m}.png')
    enhance_document(
        degraded_paths[0],
        method=m,
        output_path=out_path,
        show=True,
    )

# Demonstrate U-Net pipeline
enhance_document(
    degraded_paths[0],
    method='unet',
    output_path=os.path.join(OUTPUT_DIR, 'pipeline_unet.png'),
    unet_model=unet,
    show=True,
)

---
## Summary

| Step | Description |
|------|-------------|
| 1    | Google Drive mounted; paths configured |
| 2    | Dataset loaded from Drive; synthetic fallback generated if needed |
| 3    | Degradation simulation (Gaussian noise, salt-and-pepper, blur, motion blur) |
| 4    | Classical methods: Gaussian/Median/Bilateral/NLM filtering, Wiener, Unsharp, CLAHE, Otsu |
| 5    | U-Net deep learning model trained end-to-end with MSE + SSIM loss |
| 6    | Quantitative evaluation via PSNR and SSIM across all methods |
| 7    | Visual comparison grid and zoomed-in text crops; results saved to Drive |
| 8    | `enhance_document()` unified pipeline function |

### Tips for Best Results
- **More data is better**: train the U-Net on hundreds of paired images for production quality.
- **Tune hyperparameters**: adjust `base_filters`, `EPOCHS`, batch size, and loss `alpha` to your hardware.
- **Classical methods are fast**: use `combined` for a quick but effective baseline without training.
- **GPU runtime**: ensure you select **Runtime → Change runtime type → T4 GPU** in Colab.
- **Drive quota**: large models and datasets require sufficient Google Drive storage (free tier = 15 GB).